# Rainbow (IQN) with CartPole-v1

This notebook loads [config_rainbow.yaml](configs/config_rainbow.yaml) and trains the
`RainbowDQNAgent` on `CartPole-v1` — the Rainbow counterpart to the DQN baseline in
[exp1.ipynb](exp1.ipynb): same env, same normalisation, same training schedule, so the
two are directly comparable. Only the agent (and therefore the network wiring) changes.

Rainbow here = Double DQN + Dueling + Prioritised replay + Multi-step returns +
Distributional (**IQN**, in place of C51) + Noisy Nets. See
[src/agents/README.md](../../src/agents/README.md) for the full manual.

**Key wiring difference from the baseline.** The DQN notebook passes a whole
`QNetwork`. Rainbow instead takes only an **encoder** — the MLP trunk producing a
`(B, 128)` feature vector — and builds the dueling + noisy + IQN head *inside* the
agent. So `nn_extra_kwargs` carries encoder args only; the agent infers `feature_dim`
and `n_actions` itself.

## Imports

In [ ]:
import sys, pathlib
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

SRC = pathlib.Path.cwd().parents[2] / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from experiment import load_config, build_env, build_agent, train, make_run_logger
from agents.rainbow_agent import RainbowDQNAgent
from analysis.registry import resolve_methods
from analysis.low_rank.rank import row_rank_property_check
from analysis.visualisations.heatmaps import plot_matrix_heatmap

## Reading the config file

Note the non-default filename — this experiment has two configs (`config.yaml` for the DQN baseline, `config_rainbow.yaml` here).

In [ ]:
cfg = load_config("configs/config_rainbow.yaml")   # loads yaml, resolves device, seeds torch/numpy
print("device:", cfg["experiment"]["_device"])
cfg

## Creating the Environment

In [ ]:
env = build_env(cfg)
obs_dim = env.observation_space.shape[0]
n_actions = env.action_space.n
print("obs_dim:", obs_dim, "n_actions:", n_actions)

## Encoder (MLP trunk only)

For Rainbow the externally-supplied network is just the **encoder** — the baseline's
MLP with the Q-head removed. Everything Rainbow-specific (the cosine-τ IQN embedding,
the dueling value/advantage streams, the `NoisyLinear` layers) is built by
`RainbowDQNAgent` around it (see
[src/agents/rainbow_agent.py](../../src/agents/rainbow_agent.py)). The head width is
`agent.head_hidden` in the config.

In [ ]:
class MLPEncoder(nn.Module):
    """Maps a state (obs_dim,) -> features (B, hidden_sizes[-1]).
    RainbowDQNAgent builds the dueling + noisy + IQN head around this."""
    def __init__(self, in_dim, hidden_sizes=(128, 128)):
        super().__init__()
        layers, last = [], in_dim
        for h in hidden_sizes:
            layers += [nn.Linear(last, h), nn.ReLU()]
            last = h
        self.net = nn.Sequential(*layers)
        self.feature_dim = last

    def forward(self, x):
        return self.net(x)

## Creating the Agent

`build_agent` takes an optional `agent_cls` (default `QAgent`); pass
`RainbowDQNAgent` to build the Rainbow benchmark. `nn_extra_kwargs` carries **encoder
args only**; every Rainbow hyperparameter comes from `cfg["agent"]`.

In [ ]:
nn_extra_kwargs = {"in_dim": obs_dim, "hidden_sizes": cfg["network"]["hidden_sizes"]}   # encoder args only
agent = build_agent(cfg, env, MLPEncoder, nn_extra_kwargs, agent_cls=RainbowDQNAgent)

## Analysis (Low Rank)

The Hankel sweep runs every `ep_freq` episodes via the `analysis.hankel_sweep` config
block (dispatched inside the training loop). Because Rainbow is genuinely **dueling**,
`value_advantage` exposes learned V(s) and mean-centred A(s,·) streams — so the Hankel
**A** trace is a real signal here, not the trivially rank-deficient `Q − max Q` of the
baseline's vanilla head. Run artifacts land under `runs/<timestamp>/` when
`experiment.save_artifacts` is set.

In [ ]:
# Pass the non-default config filename so the frozen snapshot is config_rainbow.yaml.
logger = make_run_logger(cfg, config_path="configs/config_rainbow.yaml", base_dir="cached")
if logger:
    print("run artifacts ->", logger.dir)

## Agent Training

In [ ]:
rewards = train(cfg, agent, env, run_logger=logger)

## Training and Analysis Plots

In [ ]:
rewards = np.asarray(rewards, dtype=float)
plt.figure(figsize=(8, 4))
plt.plot(rewards, alpha=0.35, label="episode reward")
if len(rewards) >= 10:
    k = 10
    ma = np.convolve(rewards, np.ones(k) / k, mode="valid")
    plt.plot(range(k - 1, len(rewards)), ma, label=f"{k}-ep moving avg")
plt.xlabel("episode"); plt.ylabel("total reward"); plt.title("Rainbow (IQN) on CartPole-v1")
plt.legend()
if logger:
    plt.savefig(logger.dir / "reward_curve.png", dpi=150, bbox_inches="tight")
plt.show()

## Post-training analysis

Same as the baseline: run the generic per-matrix methods (`q_matrix_dqn`) on the final
policy, showing heatmap + spectrum for each. Hankel already ran during training.

In [ ]:
methods = resolve_methods(cfg["analysis"].get("methods", []) + cfg["analysis"].get("post_methods", []))
for method, names in methods:
    results = method(agent=agent, env=env)
    if not isinstance(results, tuple):
        results = (results,)
    for matrix, name in zip(results, names):
        print(name)
        plot_matrix_heatmap(matrix, name, save_to=logger.figure_path(f"{name} heatmap") if logger else None)
        r, sr, spk, shape, irs, ics, rc, cc, nzr, nzc = row_rank_property_check(matrix, name, save_to=logger.figure_path(name) if logger else None)
        print(f"eff_rank: {r}, stable_rank: {sr:.2f}, spikiness: {spk:.2f}, shape: {shape}, non-zero rows :{nzr}, non-zero cols:{nzc}")
        print(f"top-r leverage spread: row min={irs.min():.4g} max={irs.max():.4g} (uniform {1.0/shape[0]:.4g}) | col min={ics.min():.4g} max={ics.max():.4g} (uniform {1.0/shape[1]:.4g})")
        print(f"coherence score: row={rc:.4g} col={cc:.4g}")

## Greedy rollout video

Record one greedy (`act_greedy`) episode of the trained agent and display it inline.
The greedy action still uses the noisy weights (exploration is baked into the net).

In [ ]:
from analysis.visualisations.rollout_video import record_greedy_episode
from IPython.display import Video
import glob

video_dir = "videos"
eval_env = build_env(cfg, render_mode="rgb_array")
prefix = record_greedy_episode(agent, eval_env, video_dir, episode=0, seed=cfg["experiment"]["seed"])
mp4 = sorted(glob.glob(f"{video_dir}/{prefix}-*.mp4"))[-1]
print("saved:", mp4)
Video(mp4, embed=True)